# 02 · 映射:怎麼把一棟樓對應到"一個權利方"
上一本把多源數據拼成了一張表;**這本講那張表怎麼變成「誰算誰的」**。
方法刻意簡單:不做語意推斷、不算概率,就一張**離散、可讀、可改**的查表(`shanghai_lookup.yaml`)。

## 怎麼執行
- 點一格,按 **Shift+Enter** 執行;或選單 Run → Run All Cells 從頭跑。
- 結果與圖會顯示在該格下面。代碼都在 `engine/` 文件夾,平時不用打開。

In [ ]:
# 讓 notebook 找到 engine 裏的代碼(這格不用改,直接執行)
import sys, os
sys.path.insert(0, os.path.abspath("engine"))
import importlib, config; importlib.reload(config)
import common as C, plots
print("就緒 ·  當前站點 SLUG =", config.SLUG, "(", config.site_name(), ")")

In [ ]:
df = C.current_buildings(config.SLUG)   # 讀當前站點(隨包緩存或你建的)
print("載入 %d 棟  ·  %s" % (len(df), config.site_name()))

## 一、規則:一棟 = 一個 stakeholder,第一個來源命中即定
5 個權利方:`state` 政府/公共 · `developer` 開發商/資本 · `resident` 居民 ·
`informal` 非正式(本數據無信號、恆空) · `unknown` 未標。

**級聯 cascade**:按 `euluc → function → aoi` 順序問,**第一個給出映射的來源就定**,後面的不再問;
都不中 = `unknown`。`informal` 不在表裏(數據無信號),所以永遠不會被指派——這是誠實,不是遺漏。

In [ ]:
lk = C.load_lookup()        # 讀 shanghai_lookup.yaml
print("級聯順序:", lk["cascade"])
print("\nEULUC 用途 → 權利方(主鍵,政商可分):")
for k, v in lk["euluc"].items():
    print("   %-10s -> %s" % (k, v))
print("\n默認(都不中):", lk["default"], " · informal 不在表中(數據無信號 → 恆空)")

## 二、走查:一棟樓是怎麼被判定的
看幾棟樓的多源字段,跟着 `assign_stakeholder` 走一遍——你會看到它**在哪一級命中**。

In [ ]:
lk = C.load_lookup()
cols = [c for c in ["euluc","function","aoi_type2","aoi_type1"] if c in df.columns]
sample = df.head(6)
for _, r in sample.iterrows():
    fields = {c: C._t(r.get(c)) for c in cols}
    print("%-22s -> %s" % (str(fields), C.assign_stakeholder(r, lk)))

## 三、改這張表 = 改"誰算誰的"(反事實的一半)
學生改 `shanghai_lookup.yaml`,就是改權力地圖。下面**不動文件**、在內存裏試一條改動:
把工業用地從「開發商」改判給「政府」,看分佈怎麼變。滿意了再去真正改 YAML。

In [ ]:
import copy
lk2 = copy.deepcopy(C.load_lookup())
# 找到 euluc 里第一条映射到 developer 的用途,改判给 state(示范一次反事实)
flip = next((k for k, v in lk2["euluc"].items() if v == "developer"), None)
lk2["euluc"][flip] = "state"
a0 = C.assign_all(df).stakeholder.value_counts().to_dict()
a1 = C.assign_all(df, lk2).stakeholder.value_counts().to_dict()
print("把『%s』从 developer 改判给 state:" % flip)
print("  改前:", a0)
print("  改后:", a1)

## 四、貼出來看:權力地圖
把每棟樓按權利方著色——這就是 step2 的「權力地圖」,後面只調高度 / 上算子都建立在它上面。

In [ ]:
df = C.assign_all(df)
print("各權利方棟數:", df.stakeholder.value_counts().to_dict())
plots.power_map(df)     # 左:footprint 按角色著色;右:棟數 / 面積佔比

## 誠實邊界(映射這一關)
- `unknown` 是沒 join 到用途的樓,**不硬猜**產權。
- EULUC 面級優先 → 居住地塊裏的零星公建被併入「居民」(簡化)。
- `developer` 這裏指「資本/商業開發」的用途口徑,**不是產權考證**。
- 這是 **forward 教學練習**(改權力 → 看形態),不是規劃預測。

> 下一本:[`03 城市天際線-動手做`] —— 鎖死 footprint 與標籤,只放開高度,看權力怎麼改寫天際線。